**Setup and imports**

In [1]:
!pip install -q torchmetrics lpips transformers timm accelerate torchvision

In [16]:
!pip install -q torch-fidelity

In [2]:
import os
import zipfile
import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm
from torchvision import transforms

device = "cuda" if torch.cuda.is_available() else "cpu"

**Datasets**

In [10]:
BASE_DIR = "/content/experiments"

In [3]:
ZIP_FILES = [
    "Lora_16_florence.zip",
    "Lora_32_florence.zip",
    "Lora_16_template.zip",
    "Lora_32_template.zip",
    "Frozen_florence.zip",
    "Frozen_template.zip"
]

BASE_DIR = "/content/experiments"

os.makedirs(BASE_DIR, exist_ok=True)

for zip_file in ZIP_FILES:
    name = zip_file.replace(".zip", "")
    extract_path = os.path.join(BASE_DIR, name)
    os.makedirs(extract_path, exist_ok=True)

    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

print("All datasets extracted!")

All datasets extracted!


**Load cityscapes**

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import zipfile
import os

ZIP_PATH = "/content/drive/MyDrive/dataset trdp/leftImg8bit_trainvaltest.zip"
EXTRACT_PATH = "/content/cityscapes"

os.makedirs(EXTRACT_PATH, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Extracted to:", EXTRACT_PATH)

Extracted to: /content/cityscapes


In [12]:
def load_images_from_folder(folder, max_images=2500):
    images = []

    for root, _, files in os.walk(folder):
        for f in files:
            if f.endswith(".png"):
                images.append(os.path.join(root, f))

    return images[:max_images]

REAL_DIR = "/content/cityscapes/leftImg8bit"

real_images = load_images_from_folder(REAL_DIR, 2500)

print("Real images:", len(real_images))
print(real_images[:5])

Real images: 2500
['/content/cityscapes/leftImg8bit/train/strasbourg/strasbourg_000001_045135_leftImg8bit.png', '/content/cityscapes/leftImg8bit/train/strasbourg/strasbourg_000000_012070_leftImg8bit.png', '/content/cityscapes/leftImg8bit/train/strasbourg/strasbourg_000000_011225_leftImg8bit.png', '/content/cityscapes/leftImg8bit/train/strasbourg/strasbourg_000001_005219_leftImg8bit.png', '/content/cityscapes/leftImg8bit/train/strasbourg/strasbourg_000001_061685_leftImg8bit.png']


**Pre processor**

In [14]:
transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: (x * 255).to(torch.uint8))
])

**FID**

In [4]:
from torchmetrics.image.fid import FrechetInceptionDistance

def compute_fid(real_paths, gen_paths):
    fid = FrechetInceptionDistance(feature=2048).to(device)

    for p in tqdm(real_paths):
        img = transform(Image.open(p).convert("RGB")).unsqueeze(0).to(device)
        fid.update(img, real=True)

    for p in tqdm(gen_paths):
        img = transform(Image.open(p).convert("RGB")).unsqueeze(0).to(device)
        fid.update(img, real=False)

    return fid.compute().item()

**LPIPS**

In [5]:
import lpips

lpips_model = lpips.LPIPS(net='alex').to(device)

def compute_lpips(real_paths, gen_paths):
    scores = []

    for r, g in tqdm(zip(real_paths, gen_paths)):
        img1 = transform(Image.open(r).convert("RGB")).unsqueeze(0).to(device)
        img2 = transform(Image.open(g).convert("RGB")).unsqueeze(0).to(device)

        score = lpips_model(img1, img2)
        scores.append(score.item())

    return sum(scores) / len(scores)

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth


**CLIPscore**

In [6]:
from transformers import CLIPProcessor, CLIPModel

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def compute_clipscore(image_paths, captions):
    scores = []

    for img_path, text in tqdm(zip(image_paths, captions)):
        image = Image.open(img_path).convert("RGB")

        inputs = clip_processor(text=[text], images=image, return_tensors="pt").to(device)
        outputs = clip_model(**inputs)

        score = outputs.logits_per_image.item()
        scores.append(score)

    return sum(scores) / len(scores)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


**BLIPscore**

In [7]:
from transformers import BlipProcessor, BlipForConditionalGeneration

blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device)

def compute_blipscore(image_paths, captions):
    scores = []

    for img_path, text in tqdm(zip(image_paths, captions)):
        image = Image.open(img_path).convert("RGB")

        inputs = blip_processor(image, return_tensors="pt").to(device)
        outputs = blip_model.generate(**inputs)

        generated = blip_processor.decode(outputs[0], skip_special_tokens=True)

        # simple similarity
        score = len(set(text.split()) & set(generated.split())) / len(set(text.split()))
        scores.append(score)

    return sum(scores) / len(scores)

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

**Eval one experiment**

In [8]:
def evaluate_experiment(path):
    df = pd.read_csv(os.path.join(path, "progress.csv"))

    gen_images = [os.path.join(path, f) for f in df["image"]]
    captions = df["caption"].tolist()

    gen_images = gen_images[:2500]
    captions = captions[:2500]

    fid = compute_fid(real_images, gen_images)
    lp = compute_lpips(real_images, gen_images)
    clip = compute_clipscore(gen_images, captions)
    blip = compute_blipscore(gen_images, captions)

    return fid, lp, clip, blip

**Run all**

In [15]:
results = []

mapping = [
    ("SD 1.5 (frozen)", "Fixed", "Frozen_template"),
    ("SD 1.5 (frozen)", "Florence-2", "Frozen_florence"),

    ("LoRA Rank16", "Fixed", "Lora_16_template"),
    ("LoRA Rank16", "Florence-2", "Lora_16_florence"),

    ("LoRA Rank32", "Fixed", "Lora_32_template"),
    ("LoRA Rank32", "Florence-2", "Lora_32_florence"),
]

for model_name, prompt_type, folder in mapping:
    path = os.path.join(BASE_DIR, folder)

    print(f"\nEvaluating {folder}...")
    fid, lp, clip, blip = evaluate_experiment(path)

    results.append({
        "Model": model_name,
        "Prompt": prompt_type,
        "FID": fid,
        "LPIPS": lp,
        "CLIPScore": clip,
        "BLIPScore": blip
    })

results_df = pd.DataFrame(results)
results_df


Evaluating Frozen_template...


100%|██████████| 204/204 [00:07<00:00, 28.99it/s]
204it [00:27,  7.35it/s]
204it [00:08, 24.58it/s]
204it [00:56,  3.59it/s]



Evaluating Frozen_florence...


100%|██████████| 208/208 [00:05<00:00, 35.68it/s]
208it [00:25,  8.09it/s]
208it [00:08, 24.48it/s]
208it [00:51,  4.03it/s]



Evaluating Lora_16_template...


...


Evaluating Lora_32_template...


100%|██████████| 204/204 [00:06<00:00, 33.13it/s]
204it [00:26,  7.66it/s]
204it [00:08, 25.25it/s]
204it [00:48,  4.16it/s]



Evaluating Lora_32_florence...


100%|██████████| 348/348 [00:10<00:00, 31.97it/s]
348it [00:43,  7.94it/s]
348it [00:13, 25.78it/s]
348it [01:34,  3.67it/s]


,Model,Prompt,FID,LPIPS,CLIPScore,BLIPScore
0,SD 1.5 (frozen),Fixed,165.168701,0.382461,28.750630,0.077860
1,SD 1.5 (frozen),Florence-2,112.815369,0.396953,30.430980,0.193728
2,LoRA Rank16,Fixed,84.643143,0.232494,30.829970,0.070929
3,LoRA Rank16,Florence-2,85.517632,0.248835,28.705853,0.176318
4,LoRA Rank32,Fixed,79.065926,0.217994,31.171618,0.075749
5,LoRA Rank32,Florence-2,75.792328,0.232574,28.701071,0.188496
